# Qriton HLM — Landscape Visualization

Interactive plots of energy landscapes, basin topology, and surgery diffs.

```bash
pip install qriton-hlm[jupyter]  # includes plotly + ipywidgets
```

In [ ]:
import torch
from qriton_hlm import BasinSurgeon
from qriton_hlm.jupyter import (
    plot_landscape, plot_landscape_2d, plot_surgery_diff,
    plot_convergence, basin_explorer
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Synthetic W for demo (replace with your checkpoint)
d = 128
torch.manual_seed(0)
W = torch.randn(d, d, device=DEVICE) * 0.02
W = (W + W.T) / 2
surgeon = BasinSurgeon.from_W(W, device=DEVICE)

## Basin energy levels + population

In [ ]:
survey = surgeon.survey(layer=0)
plot_landscape(survey)

## 2D PCA projection of basin locations

In [ ]:
landscape = surgeon.landscape(layer=0)
plot_landscape_2d(landscape)

## Before / After surgery comparison

In [ ]:
before = surgeon.survey(layer=0)

surgeon.inject(layer=0, seed=42, strength=0.15)
surgeon.inject(layer=0, seed=99, strength=0.15)

after = surgeon.survey(layer=0)
plot_surgery_diff(before, after, operation='inject seed=42,99')

## Convergence trajectory

In [ ]:
# With a loaded model:
# trace = surgeon.trace(layer=5, text='Hello world')
# plot_convergence(trace['trajectory'])

# Manual trace for demo
from qriton_hlm.core import _converge, poly_interaction, compute_energy

W = surgeon.get_W(0)
x = torch.randn(W.shape[0], device=DEVICE) * 0.5
trajectory = []
for t in range(100):
    tau = 0.9 + (0.1 - 0.9) * (t / 100)
    fx = poly_interaction(x, 3)
    h = W @ fx
    x_new = (1 - tau) * x + tau * torch.tanh(7.0 * h)
    trajectory.append({'step': t, 'energy': compute_energy(x_new, W),
                       'delta': (x_new - x).norm().item()})
    if (x_new - x).norm().item() / (x.norm().item() + 1e-8) < 5e-3:
        break
    x = x_new

plot_convergence(trajectory)

## Interactive explorer (ipywidgets)

In [ ]:
surgeon.restore(0)
basin_explorer(surgeon, layer=0)

---
**Next:** `04_model_comparison.ipynb` — compare basins between two models